# TensorFlow Logistic Regression Baseline
Predicts `blueWins` from 10-minute features using a single-layer sigmoid classifier.

## Why this model?
- Logistic regression is the simplest interpretable classifier and satisfies the course requirement for a linear baseline.
- With standardized numeric features, the learned weights translate directly into statements such as "every 1k gold swing increases win odds by X%".
- TensorFlow implementation keeps the tooling consistent across all models.

### Hyperparameters we care about
| Hyperparameter | Why? | Typical tuning range |
| --- | --- | --- |
| L2 penalty (`kernel_regularizer`) | Prevents coefficient blow-up on correlated gold/XP stats. | `1e-5` – `1e-2` |
| Learning rate | Governs convergence speed vs. stability. | `5e-4` – `5e-3` |
| Batch size | Balances noisy gradients vs. throughput. | `32`, `64`, `128` |
| Epochs / EarlyStopping patience | Stops once validation ROC-AUC plateaus to avoid overfit. | 30–150 epochs, patience 5–15 |


In [1]:
from pathlib import Path
import sys

import numpy as np
import pandas as pd
import tensorflow as tf
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

# Locate repo root even if notebook launches from notebooks/modeling
REPO_ROOT = Path.cwd().resolve()
while not (REPO_ROOT / 'src').exists():
    if REPO_ROOT == REPO_ROOT.parent:
        raise RuntimeError('Could not find project root containing src/')
    REPO_ROOT = REPO_ROOT.parent

print(f'Using repo root: {REPO_ROOT}')
DATA_PATH = REPO_ROOT / 'data/raw/high_diamond_ranked_10min.csv'
RAW = pd.read_csv(DATA_PATH)
TARGET = 'blueWins'
FEATURES = [col for col in RAW.columns if col not in {TARGET, 'gameId'}]

X_train, X_test, y_train, y_test = train_test_split(
    RAW[FEATURES], RAW[TARGET], test_size=0.2, stratify=RAW[TARGET], random_state=42
)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)
input_dim = X_train_scaled.shape[1]
print(f'Train shape: {X_train_scaled.shape}, Test shape: {X_test_scaled.shape}')


Using repo root: /Users/liamsandy/ML_Project
Train shape: (7903, 38), Test shape: (1976, 38)


### Load engineered features (reuse across models)

In [ ]:
import pandas as pd

def safe_ratio(numerator, denominator, fill_value=0.0):
    denominator = denominator.replace(0, pd.NA)
    return (numerator / denominator).fillna(fill_value)

def engineer_features_inline(df):
    features = df.copy()
    blue_obj = features['blueDragons'] + features['blueHeralds'] + features['blueEliteMonsters']
    red_obj = features['redDragons'] + features['redHeralds'] + features['redEliteMonsters']
    total_obj = blue_obj + red_obj
    features['blue_objective_share'] = safe_ratio(blue_obj, total_obj, 0.5)
    features['red_objective_share'] = safe_ratio(red_obj, total_obj, 0.5)
    total_wards = features['blueWardsPlaced'] + features['redWardsPlaced']
    features['blue_vision_share'] = safe_ratio(features['blueWardsPlaced'], total_wards, 0.5)
    features['red_vision_share'] = safe_ratio(features['redWardsPlaced'], total_wards, 0.5)
    features['blue_ward_efficiency'] = safe_ratio(features['blueWardsDestroyed'], features['blueWardsPlaced'], 0.0)
    features['red_ward_efficiency'] = safe_ratio(features['redWardsDestroyed'], features['redWardsPlaced'], 0.0)
    total_kills = features['blueKills'] + features['redKills']
    features['blue_kill_share'] = safe_ratio(features['blueKills'], total_kills, 0.5)
    features['red_kill_share'] = safe_ratio(features['redKills'], total_kills, 0.5)
    features['blue_gold_per_kill'] = safe_ratio(features['blueTotalGold'], features['blueKills'] + 1)
    features['red_gold_per_kill'] = safe_ratio(features['redTotalGold'], features['redKills'] + 1)
    features['blue_xp_per_min'] = features['blueTotalExperience'] / 10.0
    features['red_xp_per_min'] = features['redTotalExperience'] / 10.0
    features['gold_obj_momentum'] = features['blueGoldDiff'] * (features['blueDragons'] + features['blueHeralds'])
    features['xp_kill_momentum'] = features['blueExperienceDiff'] * features['blueKills']
    features['gold_diff_per_min'] = features['blueGoldDiff'] / 10.0
    features['xp_diff_per_min'] = features['blueExperienceDiff'] / 10.0
    if 'gameId' in features:
        features = features.drop(columns=['gameId'])
    return features

engineered_csv = (REPO_ROOT / 'data/processed/engineered_features.csv').resolve()
if engineered_csv.exists():
    engineered_df = pd.read_csv(engineered_csv)
    print('Loaded engineered CSV:', engineered_df.shape)
else:
    print('Engineered CSV missing; computing inline.')
    engineered_df = engineer_features_inline(RAW)
    print('Inline engineered shape:', engineered_df.shape)


In [18]:
from pathlib import Path
import pandas as pd

ENGINEERED_PATH = (REPO_ROOT / 'data/processed/engineered_features.csv').resolve()
if ENGINEERED_PATH.exists():
    engineered_df = pd.read_csv(ENGINEERED_PATH)
else:
    print('Engineered CSV not found; falling back to on-the-fly features.')
    from feature_engineering_playbook import engineer_features  # adjust if needed
    engineered_df = engineer_features(BASE_DF)
print('Engineered shape:', engineered_df.shape)


Engineered shape: (9879, 55)


## Build the TensorFlow logistic model
Single dense neuron with sigmoid activation = classic logistic regression.

In [4]:
def build_logistic_model(input_dim: int, l2: float = 5e-4, learning_rate: float = 1e-3):
    model = tf.keras.Sequential([
        tf.keras.layers.Input(shape=(input_dim,)),
        tf.keras.layers.Dense(1, activation='sigmoid', kernel_regularizer=tf.keras.regularizers.l2(l2)),
    ])
    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=learning_rate),
        loss='binary_crossentropy', # heavily penalizes wrong guesses
        metrics=[
            tf.keras.metrics.BinaryAccuracy(name='accuracy'),
            tf.keras.metrics.AUC(name='roc_auc'),
        ],
    )
    return model

log_model = build_logistic_model(input_dim)
log_model.summary()


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense (Dense)                   │ (None, 1)              │            39 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 39 (156.00 B)

 Trainable params: 39 (156.00 B)

 Non-trainable params: 0 (0.00 B)

## Training template
Tune `learning_rate`, `l2`, batch size, and `EarlyStopping` patience to explore generalization.

In [6]:
callbacks = [
    tf.keras.callbacks.EarlyStopping(
        monitor='val_roc_auc',
        mode='max',
        patience=10,
        restore_best_weights=True,
    )
]
history = log_model.fit(
    X_train_scaled,
    y_train,
    validation_split=0.15,
    epochs=120,
    batch_size=64,
    callbacks=callbacks,
    verbose=1,
)


Epoch 1/120
105/105 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.5760 - loss: 0.7632 - roc_auc: 0.6131 - val_accuracy: 0.6433 - val_loss: 0.6825 - val_roc_auc: 0.6922
Epoch 2/120
105/105 ━━━━━━━━━━━━━━━━━━━━ 0s 762us/step - accuracy: 0.6699 - loss: 0.6248 - roc_auc: 0.7369 - val_accuracy: 0.6931 - val_loss: 0.6203 - val_roc_auc: 0.7463
Epoch 3/120
105/105 ━━━━━━━━━━━━━━━━━━━━ 0s 764us/step - accuracy: 0.7006 - loss: 0.5800 - roc_auc: 0.7723 - val_accuracy: 0.7015 - val_loss: 0.5881 - val_roc_auc: 0.7678
Epoch 4/120
105/105 ━━━━━━━━━━━━━━━━━━━━ 0s 769us/step - accuracy: 0.7149 - loss: 0.5576 - roc_auc: 0.7879 - val_accuracy: 0.7057 - val_loss: 0.5695 - val_roc_auc: 0.7820
Epoch 5/120
105/105 ━━━━━━━━━━━━━━━━━━━━ 0s 762us/step - accuracy: 0.7228 - loss: 0.5454 - roc_auc: 0.7977 - val_accuracy: 0.7066 - val_loss: 0.5585 - val_roc_auc: 0.7897
Epoch 6/120
105/105 ━━━━━━━━━━━━━━━━━━━━ 0s 748us/step - accuracy: 0.7265 - loss: 0.5388 - roc_auc: 0.8029 - val_accuracy: 0.7150 - val_loss: 0.552

In [8]:
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score

test_probs = log_model.predict(X_test_scaled).flatten()
test_preds = (test_probs >= 0.5).astype(int)
print('Test Accuracy:', accuracy_score(y_test, test_preds))
print('Test F1:', f1_score(y_test, test_preds))
print('Test ROC-AUC:', roc_auc_score(y_test, test_probs))


62/62 ━━━━━━━━━━━━━━━━━━━━ 0s 517us/step
Test Accuracy: 0.7160931174089069
Test F1: 0.7165234967155129
Test ROC-AUC: 0.805586288852009


## Engineered feature experiment (logistic baseline)

In [20]:
engineered_features = [col for col in engineered_df.columns if col != TARGET]
X_train_eng, X_test_eng, y_train_eng, y_test_eng = train_test_split(
    engineered_df[engineered_features],
    engineered_df[TARGET],
    test_size=0.2,
    stratify=engineered_df[TARGET],
    random_state=42,
)
scaler_eng = StandardScaler()
X_train_eng_scaled = scaler_eng.fit_transform(X_train_eng)
X_test_eng_scaled = scaler_eng.transform(X_test_eng)
eng_model = build_logistic_model(X_train_eng_scaled.shape[1])
eng_history = eng_model.fit(
    X_train_eng_scaled,
    y_train_eng,
    validation_split=0.15,
    epochs=120,
    batch_size=64,
    callbacks=callbacks,
    verbose=0,
)
eng_probs = eng_model.predict(X_test_eng_scaled).flatten()
eng_preds = (eng_probs >= 0.5).astype(int)
print('Engineered Accuracy:', accuracy_score(y_test_eng, eng_preds))
print('Engineered F1:', f1_score(y_test_eng, eng_preds))
print('Engineered ROC-AUC:', roc_auc_score(y_test_eng, eng_probs))


62/62 ━━━━━━━━━━━━━━━━━━━━ 0s 435us/step
Engineered Accuracy: 0.6902834008097166
Engineered F1: 0.6851851851851852
Engineered ROC-AUC: 0.7689829327760362


In [10]:
log_model.layers[0].kernel

<Variable path=sequential/dense/kernel, shape=(38, 1), dtype=float32, value=[[-0.04202228]
 [ 0.00580314]
 [ 0.07336096]
 [-0.14580522]
 [ 0.17578204]
 [-0.01966345]
 [ 0.12553093]
 [ 0.05914853]
 [-0.08707745]
 [-0.00957292]
 [ 0.05556294]
 [ 0.0924422 ]
 [ 0.00661485]
 [-0.27833992]
 [ 0.02574523]
 [ 0.05548052]
 [ 0.16041343]
 [ 0.20592208]
 [ 0.32676488]
 [-0.00762517]
 [-0.00982479]
 [ 0.04033615]
 [-0.17664818]
 [ 0.07463816]
 [ 0.07046938]
 [-0.05161964]
 [-0.0987516 ]
 [-0.01038388]
 [ 0.08849192]
 [-0.24643604]
 [ 0.00128071]
 [-0.19257572]
 [ 0.11712333]
 [ 0.05085249]
 [-0.54688585]
 [-0.05969334]
 [-0.01193076]
 [-0.05334469]]>

### Interpretation checklist
- Inspect `log_model.layers[0].kernel` after training to see which features drive wins.
- Because the model is linear, SHAP/coefficients are sufficient for the report.

- Accuracy ≈0.716: the logistic baseline correctly flags about 72% of held-out
  matches using only 10-minute stats—much better than a 50/50 coin flip.
  - F1 ≈0.717: precision and recall on blue-win predictions stay balanced, so
  the model isn’t just chasing one metric at the expense of the other.
  - ROC-AUC ≈0.806: strong separation between blue-win and red-win classes; the
  ranking ability passes the 0.80 target from the plan, suggesting the linear
  model already extracts meaningful signal from gold/XP/objective features.